# Email Commands Walkthrough

`email_commands.py` is security-sensitive -- this notebook demonstrates the parsing,
validation, and fail-safe behavior interactively, so you can build real confidence in it
before enabling it against a live inbox. Nothing here connects to a real mail server; every
cell calls `parse_command()` directly with hand-written email bodies.

See `docs/EMAIL_COMMANDS.md` for the full security model and syntax reference.

In [ ]:
# Package is pip-installed editable, no sys.path hacking needed
from momentum_trading.interfaces.email_commands import parse_command, build_reply_body, log_command_attempt, ADJUSTABLE_PARAMS

TRUSTED = "trader@example.com"

## 1. The most important property: sender authentication

Only emails from the exact configured `trusted_sender` are ever parsed -- everything else is
rejected before the body is even inspected, regardless of how well-formed it looks.

In [ ]:
# A perfectly valid-looking command from the WRONG sender
result = parse_command("attacker@evil.com", TRUSTED, "ACTION: PAUSE\nPORTFOLIO: ALL")
print("Success:", result.success)
print("Error:", result.error)

## 2. Valid simple commands

In [ ]:
for action in ["PAUSE", "RESUME", "SKIP_NEXT_REBALANCE", "TRIGGER_REPORT", "STATUS"]:
    r = parse_command(TRUSTED, TRUSTED, f"ACTION: {action}\nPORTFOLIO: portfolio1")
    print(f"{action}: success={r.success}")

## 3. LIQUIDATE: extra friction for the most destructive command

Requires an EXACT confirmation phrase, not just the action word.

In [ ]:
print("--- No confirmation ---")
r = parse_command(TRUSTED, TRUSTED, "ACTION: LIQUIDATE\nPORTFOLIO: portfolio1")
print("Success:", r.success)

print("\n--- Wrong phrase ---")
r = parse_command(TRUSTED, TRUSTED, "ACTION: LIQUIDATE\nPORTFOLIO: portfolio1\nCONFIRM: yes do it")
print("Success:", r.success)

print("\n--- Correct phrase ---")
r = parse_command(TRUSTED, TRUSTED, "ACTION: LIQUIDATE\nPORTFOLIO: portfolio1\nCONFIRM: I confirm liquidation")
print("Success:", r.success)

## 4. ADJUST_PARAM: the allowlist (critical security boundary)

Only these fields, with these bounds, can EVER be changed via email:

In [ ]:
print("Allowlisted parameters:", ADJUSTABLE_PARAMS)

print("\n--- Allowlisted field, in bounds: accepted ---")
r = parse_command(TRUSTED, TRUSTED, "ACTION: ADJUST_PARAM\nPORTFOLIO: portfolio1\nPARAM: stop_loss_pct\nVALUE: 0.15")
print("Success:", r.success)

print("\n--- NON-allowlisted field: rejected (this is the critical test) ---")
r = parse_command(TRUSTED, TRUSTED, "ACTION: ADJUST_PARAM\nPORTFOLIO: portfolio1\nPARAM: initial_capital\nVALUE: 999999")
print("Success:", r.success)
print("Error:", r.error)

## 5. SET_MAX_DRAWDOWN: tightening-only

Parses and validates the requested value here; the "can only tighten, never loosen" rule is
enforced separately in `daily_runner.py` at the point the value is actually used (it needs the
live config to compare against, which isn't available at parse time).

In [ ]:
r = parse_command(TRUSTED, TRUSTED, "ACTION: SET_MAX_DRAWDOWN\nPORTFOLIO: portfolio1\nVALUE: 0.10")
print("Valid fraction:", r.success, r.command.new_value if r.success else r.error)

r = parse_command(TRUSTED, TRUSTED, "ACTION: SET_MAX_DRAWDOWN\nPORTFOLIO: portfolio1\nVALUE: 1.5")
print("Out of (0,1) range:", r.success)

## 6. Fail-safe behavior on malformed input

Nothing here should ever raise -- an unrecognized action or garbage body is rejected cleanly.

In [ ]:
for body in ["ACTION: DELETE_EVERYTHING\nPORTFOLIO: ALL", "this is not a command at all", ""]:
    r = parse_command(TRUSTED, TRUSTED, body)
    print(f"Input: {body[:40]!r:42} -> success={r.success}")

## 7. Reply bodies and audit logging

In [ ]:
ok = parse_command(TRUSTED, TRUSTED, "ACTION: RESUME\nPORTFOLIO: portfolio1")
bad = parse_command("evil@x.com", TRUSTED, "ACTION: PAUSE\nPORTFOLIO: portfolio1")

print("--- Accepted reply ---")
print(build_reply_body(ok))
print("\n--- Rejected reply ---")
print(build_reply_body(bad))

# Every attempt -- accepted or rejected -- is logged to a hash-chained audit trail
log_command_attempt(TRUSTED, ok, log_path="data/demo_commands_log.csv")
log_command_attempt("evil@x.com", bad, log_path="data/demo_commands_log.csv")

import pandas as pd
display(pd.read_csv("data/demo_commands_log.csv"))

## Going live (not this notebook)

Real IMAP polling (`poll_and_process_commands()`) requires real credentials and is not
demonstrated here -- see `docs/EMAIL_COMMANDS.md` for setup, and note that function has NOT been
tested against a real mail server in this project. Test it against a real dedicated inbox
before relying on it.